In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import TensorDataset,DataLoader,Subset
from PIL import Image
import kagglehub

Download the dataset

In [ ]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")

print("path",path)

In [ ]:
from torch.accelerator import is_available
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using gpu {device}")

In [ ]:
TRAIN_PATH = os.path.join("PlantVillage","train")
VAL_PATH = os.path.join("PlantVillage","val")

# Transformation

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3 ,std=[0.5]*3)
    ]
)

# CustomDataset

In [ ]:
from IPython.lib.display import isfile
from posix import listdir
class MultiClassClassfication(TensorDataset):
  def __init__(self,root_dir,transform=None):
    super().__init__()

    self.samples = []
    self.transform = transform
    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

    self.class_to_idx = {cls_name : idx for idx,cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      cls_path = os.path.join(root_dir,class_name)

      for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path,img_name)

        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append((img_path,label))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self,index):

    img_path,label = self.samples[index]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image,label


# Load the Dataset

In [ ]:
import os
print(os.listdir("/kaggle/input/plantvillage"))

In [ ]:
TRAIN_PATH = os.path.join(path, "PlantVillage", "train")
VAL_PATH = os.path.join(path, "PlantVillage", "val")
train_dataset_full = MultiClassClassfication(TRAIN_PATH,transform)
test_dataset_full = MultiClassClassfication(VAL_PATH, transform)
num_classes = len(train_dataset_full.classes)

In [ ]:
train_dataset = Subset(train_dataset_full,list(range(min(20000,len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full,list(range(min(20000,len(test_dataset_full)))))

In [ ]:
pin = True if device.type == 'cuda' else False

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=pin)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=pin)

# CNN

In [ ]:
class MyCNN(nn.Module):
  def __init__(self,num_classes) :
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),

        nn.Conv2d(32,64,kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),

        nn.Conv2d(64,128, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2)

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*16*16, 128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(64,num_classes)
    )

  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [ ]:
model = MyCNN(num_classes=num_classes).to(device)

# Training

In [ ]:
learning_rate = 0.001
epochs =10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = learning_rate)

In [ ]:
for epoch in range(epochs):
  model.train()
  total_loss = 0
  for batch_features,batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)

    loss = criterion(outputs,batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  avg_loss = total_loss /len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Loss : {avg_loss:.4f}")

# Evaluate the model

In [ ]:
def evaluate(loader):
  model.eval()
  total =  0
  correct = 0

  with torch.no_grad():
    for batch_features,batch_labels in loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      outputs = model(batch_features)
      _,predicted = torch.max(outputs,1)
      total += batch_labels.size(0)
      correct += (predicted == batch_labels).sum().item()
  return correct / total

In [ ]:
test_acc = evaluate(test_loader)
print("Test Accuracy:",test_acc)